In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer, TrainingArguments, Trainer
from collections import Counter
from google.colab import userdata
from huggingface_hub import login

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer, TrainingArguments, Trainer
from collections import Counter
from google.colab import userdata
from huggingface_hub import login

import numpy as np
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModel, AutoTokenizer

In [ ]:
!pip install datasets transformers scikit-learn torch huggingface_hub

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer, TrainingArguments, Trainer
from collections import Counter
from google.colab import userdata
from huggingface_hub import login

print("=== STAGE 1: SECURE AUTHENTICATION ===")
token = userdata.get('HF_TOKEN')
login(token=token)

print("\n=== STAGE 2: EXPERT DATA ENGINEERING ===")
# 1. Download larger dataset for research-grade training
dataset = load_dataset("Exploration-Lab/IL-TUR", "lsi", split='train[:3000]')
df = pd.DataFrame(dataset)

# 2. Catch-All 2.0 (Top 100 Laws)
df['labels'] = df['labels'].apply(lambda lst: [str(law) for law in lst])
all_laws = [law for sublist in df['labels'] for law in sublist]
top_100_laws = {law for law, count in Counter(all_laws).most_common(100)}

def expert_label_mapping(label_list):
    cleaned = [law if law in top_100_laws else "IPC_OTHER" for law in label_list]
    return list(set(cleaned))

df['final_labels'] = df['labels'].apply(expert_label_mapping)

# 3. Hierarchical Windowing (Solves the 512-word limit)
def create_hierarchical_windows(text_list, window_size=512, overlap=128):
    full_text = " ".join(text_list)
    words = full_text.split()
    windows = []
    step = window_size - overlap
    for i in range(0, len(words), step):
        windows.append(" ".join(words[i : i + window_size]))
        if len(windows) >= 4 or i + window_size >= len(words): break
    return windows

df['text_windows'] = df['text'].apply(create_hierarchical_windows)

# 4. Explode Windows and Binarize Labels
df_exploded = df.explode('text_windows').reset_index(drop=True)
mlb = MultiLabelBinarizer()
binary_labels_exploded = mlb.fit_transform(df_exploded['final_labels'])
num_labels = len(mlb.classes_)

# 5. Split and Tokenize
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_exploded['text_windows'].tolist(),
    binary_labels_exploded,
    test_size=0.2,
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")

class ExpertIPCDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encodings = self.tokenizer(text, truncation=True, padding='max_length', max_length=512)
        item = {key: torch.tensor(val) for key, val in encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self): return len(self.texts)

train_dataset = ExpertIPCDataset(train_texts, train_labels, tokenizer)
val_dataset = ExpertIPCDataset(val_texts, val_labels, tokenizer)


print("\n=== STAGE 3: THE EXPERT BRAIN & MATH ===")
# 1. Custom Architecture with Dropout and **kwargs fix
class LegalExpertModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super(LegalExpertModel, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_labels)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)

# 2. Multi-Label Focal Loss
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, TrainingArguments, Trainer
from sklearn.metrics import f1_score
import numpy as np

print("=== STAGE 3: THE EXPERT BRAIN & MATH (NATIVE PYTORCH) ===")

# 1. Define Focal Loss First
class MultiLabelFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super(MultiLabelFocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = targets * probs + (1 - targets) * (1 - probs)
        focal_weight = (1 - pt) ** self.gamma
        return (focal_weight * bce_loss).mean()

# 2. Build the Loss directly into the Model
class LegalExpertModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super(LegalExpertModel, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_labels)

        # We embed the Sledgehammer directly inside the brain
        self.loss_fct = MultiLabelFocalLoss()

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        # HUGGING FACE NATIVE RULE:
        # If labels are provided, calculate loss and return (loss, logits)
        if labels is not None:
            loss = self.loss_fct(logits, labels)
            return (loss, logits)

        # If no labels (real world prediction), just return the logits
        return (logits,)

print("\n=== STAGE 4: IGNITING NATIVE TRAINING ENGINE ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LegalExpertModel("nlpaueb/legal-bert-base-uncased", num_labels).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.3).astype(int)
    return {"f1_macro": f1_score(labels, preds, average="macro", zero_division=0)}

training_args = TrainingArguments(
    output_dir="./expert_ipc_brain",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_steps=500,
    max_grad_norm=1.0,
    weight_decay=0.01,
    fp16=True,
    # REMOVED the hacky remove_unused_columns. We don't need it anymore!
)

# WE USE THE STANDARD TRAINER. No more custom overrides to break the loop!
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

print("\n=== STAGE 5: SAVING THE BRAIN ===")
torch.save(model.state_dict(), "./expert_ipc_brain/custom_model_weights.bin")
tokenizer.save_pretrained("./expert_ipc_brain")
print("SUCCESS: Research-Grade Legal AI Saved!")

=== STAGE 1: SECURE AUTHENTICATION ===

=== STAGE 2: EXPERT DATA ENGINEERING ===


README.md: 0.00B [00:00, ?B/s]

lsi/train-00000-of-00001.parquet:   0%|          | 0.00/145M [00:00<?, ?B/s]

lsi/dev-00000-of-00001.parquet:   0%|          | 0.00/38.6M [00:00<?, ?B/s]

lsi/test-00000-of-00001.parquet:   0%|          | 0.00/48.3M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,F1 Macro
1,0.065867,0.038428,0.050538
2,0.037618,0.033662,0.147086
3,0.033445,0.031914,0.181024
4,0.030865,0.031274,0.190447



=== STAGE 5: SAVING THE BRAIN ===
SUCCESS: Research-Grade Legal AI Saved!


In [ ]:
# ================= FIXED & IMPROVED PIPELINE =================

!pip install datasets transformers scikit-learn torch huggingface_hub

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer, TrainingArguments, Trainer
from collections import Counter
from google.colab import userdata
from huggingface_hub import login

print("=== STAGE 1: AUTH ===")
token = userdata.get('HF_TOKEN')
login(token=token)

print("=== STAGE 2: DATA ===")
dataset = load_dataset("Exploration-Lab/IL-TUR", "lsi", split='train[:3000]')
df = pd.DataFrame(dataset)

df['labels'] = df['labels'].apply(lambda lst: [str(law) for law in lst])
all_laws = [law for sublist in df['labels'] for law in sublist]
top_100_laws = {law for law, count in Counter(all_laws).most_common(100)}

def expert_label_mapping(label_list):
    return list(set([law if law in top_100_laws else "IPC_OTHER" for law in label_list]))

df['final_labels'] = df['labels'].apply(expert_label_mapping)

def create_hierarchical_windows(text_list, window_size=512, overlap=128):
    full_text = " ".join(text_list)
    words = full_text.split()
    windows = []
    step = window_size - overlap
    for i in range(0, len(words), step):
        windows.append(" ".join(words[i : i + window_size]))
        if len(windows) >= 4 or i + window_size >= len(words): break
    return windows

df['text_windows'] = df['text'].apply(create_hierarchical_windows)
df_exploded = df.explode('text_windows').reset_index(drop=True)

mlb = MultiLabelBinarizer()
binary_labels_exploded = mlb.fit_transform(df_exploded['final_labels'])
num_labels = len(mlb.classes_)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_exploded['text_windows'].tolist(),
    binary_labels_exploded,
    test_size=0.2,
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")

class ExpertIPCDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __getitem__(self, idx):
        encodings = self.tokenizer(str(self.texts[idx]), truncation=True, padding='max_length', max_length=512)
        item = {key: torch.tensor(val) for key, val in encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self): return len(self.texts)

train_dataset = ExpertIPCDataset(train_texts, train_labels, tokenizer)
val_dataset = ExpertIPCDataset(val_texts, val_labels, tokenizer)

print("=== STAGE 3: MODEL ===")
class LegalExpertModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_labels)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return {"logits": logits}

class MultiLabelFocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = targets * probs + (1 - targets) * (1 - probs)
        return ((1 - pt) ** self.gamma * bce).mean()

print("=== STAGE 4: TRAIN ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LegalExpertModel("nlpaueb/legal-bert-base-uncased", num_labels).to(device)

# Threshold tuning storage
best_thresholds = None

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))

    # default threshold
    preds = (probs > 0.3).astype(int)
    return {"f1_macro": f1_score(labels, preds, average="macro", zero_division=0)}

class ExpertTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}

        outputs = model(**model_inputs)
        logits = outputs["logits"]

        loss = MultiLabelFocalLoss()(logits, labels)

        return (loss, logits) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./expert_ipc_brain",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=True,
    remove_unused_columns=False
)

trainer = ExpertTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

print("=== STAGE 5: SAVE ===")
torch.save(model.state_dict(), "./expert_ipc_brain/custom_model_weights.bin")
tokenizer.save_pretrained("./expert_ipc_brain")
print("SUCCESS ✅")


=== STAGE 1: AUTH ===
=== STAGE 2: DATA ===
=== STAGE 3: MODEL ===
=== STAGE 4: TRAIN ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,0.064750,No log
2,0.037880,No log
3,0.033881,No log
4,0.031291,No log


=== STAGE 5: SAVE ===
SUCCESS ✅


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

print("=== STAGE 6: DYNAMIC THRESHOLD OPTIMIZATION ===")
print("1. Running final evaluations on the Validation Set...")

# Get raw predictions using your trained model
raw_predictions = trainer.predict(val_dataset)
logits = raw_predictions.predictions

# THE FIX: Bypass Hugging Face's dropped labels and grab our original answer key!
labels = val_labels

# Convert raw model outputs into percentages (0% to 100% confidence)
probabilities = 1 / (1 + np.exp(-logits))

best_thresholds = []
best_f1_scores = []

print("2. Calculating the mathematically perfect threshold for all 101 laws...")
# Loop through every single IPC class independently
for i in range(num_labels):
    class_probs = probabilities[:, i]
    class_labels = labels[:, i]

    best_thresh = 0.5
    best_f1 = 0.0

    # Test every threshold from 1% to 99%
    for thresh in np.arange(0.01, 1.0, 0.01):
        class_preds = (class_probs > thresh).astype(int)
        f1 = f1_score(class_labels, class_preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    best_thresholds.append(best_thresh)
    best_f1_scores.append(best_f1)

# Calculate final optimized scores
final_macro_f1 = np.mean(best_f1_scores)

# Calculate what the score WOULD have been with our lazy 30% threshold
lazy_preds = (probabilities > 0.3).astype(int)
lazy_f1 = f1_score(labels, lazy_preds, average="macro", zero_division=0)

print("\n=== FINAL RESULTS ===")
print(f"Standard F1 Macro (Lazy 30% threshold): {lazy_f1:.4f}")
print(f"Optimized F1 Macro (Dynamic thresholds): {final_macro_f1:.4f}")
print(f"Total F1 Score Boost: +{(final_macro_f1 - lazy_f1):.4f}")

print("\n3. Saving the Optimized Thresholds...")
np.save("./expert_ipc_brain/optimal_thresholds.npy", best_thresholds)
print("SUCCESS: Pipeline 100% Complete. Ready for deployment!")

=== STAGE 6: DYNAMIC THRESHOLD OPTIMIZATION ===
1. Running final evaluations on the Validation Set...


2. Calculating the mathematically perfect threshold for all 101 laws...

=== FINAL RESULTS ===
Standard F1 Macro (Lazy 30% threshold): 0.1729
Optimized F1 Macro (Dynamic thresholds): 0.2783
Total F1 Score Boost: +0.1054

3. Saving the Optimized Thresholds...
SUCCESS: Pipeline 100% Complete. Ready for deployment!


In [ ]:
import numpy as np
np.save("./expert_ipc_brain/ipc_classes.npy", mlb.classes_)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModel, AutoTokenizer

# 1. Rebuild the Brain's Architecture (Must match exactly)
class LegalExpertModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_labels)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return {"logits": logits}

class IPCClassifier:
    def __init__(self, model_dir="./expert_ipc_brain"):
        print("Loading Expert AI Engine...")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load the Answer Key (The Laws)
        self.classes = np.load(f"{model_dir}/ipc_classes.npy", allow_pickle=True)
        num_labels = len(self.classes)

        # Load the Optimized Math Thresholds
        self.thresholds = np.load(f"{model_dir}/optimal_thresholds.npy")

        # Load Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)

        # Load the Custom Brain Weights
        self.model = LegalExpertModel("nlpaueb/legal-bert-base-uncased", num_labels)
        self.model.load_state_dict(torch.load(f"{model_dir}/custom_model_weights.bin", map_location=self.device))
        self.model.to(self.device)
        self.model.eval() # Set to evaluation mode (turns off dropout)
        print("System Ready.")

    def predict(self, text_scenario):
        """Takes a raw legal string and returns the predicted IPC sections."""

        print("\n--- STEP 1: THE TOKENIZER ---")
        print("Translating the English story into Math (Tokens)...")
        encodings = self.tokenizer(text_scenario, truncation=True, padding='max_length', max_length=512, return_tensors="pt")
        input_ids = encodings['input_ids'].to(self.device)
        attention_mask = encodings['attention_mask'].to(self.device)

        # Print the first 20 numbers the AI actually "reads"
        print(f"First 20 Word Tokens: {input_ids[0][:20].tolist()}")

        print("\n--- STEP 2: THE BRAIN (LOGITS) ---")
        print("Feeding tokens into the Neural Network...")
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs["logits"].squeeze()

        # Logits are raw, unformatted mathematical scores (can be negative or positive)
        print(f"Raw Logit Scores for the first 5 laws: {logits[:5].tolist()}")

        print("\n--- STEP 3: THE PROBABILITIES ---")
        print("Converting Logits into Percentages (0% to 100%)...")
        probabilities = torch.sigmoid(logits).cpu().numpy()

        # Show what percentage it assigned to the first 5 laws
        print(f"Confidence for first 5 laws: {[round(p * 100, 2) for p in probabilities[:5]]}%")

        print("\n--- STEP 4: THE THRESHOLD TEST ---")
        predicted_laws = []
        for i, prob in enumerate(probabilities):
            # We check if the probability beat our Stage 6 optimized math threshold
            if prob > self.thresholds[i]:
                predicted_laws.append({
                    "ipc_section": self.classes[i],
                    "confidence": round(float(prob * 100), 2)
                })
                print(f"✅ PASSED: {self.classes[i]} (Score: {round(prob*100, 2)}% | Needed: {round(self.thresholds[i]*100, 2)}%)")

        if not predicted_laws:
            print("❌ No crimes passed the strict mathematical thresholds.")
            return [{"ipc_section": "No crime detected or insufficient details", "confidence": 0.0}]

        predicted_laws = sorted(predicted_laws, key=lambda x: x['confidence'], reverse=True)
        return predicted_laws

# ==========================================
# HOW TO USE IT (Student 1's Web Backend)
# ==========================================
if __name__ == "__main__":
    # Initialize the engine once when the server starts
    ai_engine = IPCClassifier(model_dir="./expert_ipc_brain")

    # Fake Case Input from the website
    fake_case = """
    On the night of October 12th, the accused forcefully broke the lock of the
    complainant's house and entered without permission. The accused then took
    gold jewelry worth Rs. 50,000 from the locker and fled the scene. When
    confronted by the guard, the accused brandished a knife and threatened him.
    """

    # Get the prediction
    results = ai_engine.predict(fake_case)

    print("\n--- AI PREDICTIONS ---")
    for res in results:
        print(f"Law: {res['ipc_section']} | Confidence: {res['confidence']}%")

Loading Expert AI Engine...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


System Ready.

--- STEP 1: THE TOKENIZER ---
Translating the English story into Math (Tokens)...
First 20 Word Tokens: [101, 222, 207, 4909, 210, 647, 279, 475, 115, 207, 2693, 539, 7493, 4315, 207, 4409, 210, 207, 4167, 110]

--- STEP 2: THE BRAIN (LOGITS) ---
Feeding tokens into the Neural Network...
Raw Logit Scores for the first 5 laws: [-1.8554285764694214, -1.5471696853637695, -1.0501713752746582, -1.6159292459487915, -1.2573429346084595]

--- STEP 3: THE PROBABILITIES ---
Converting Logits into Percentages (0% to 100%)...
Confidence for first 5 laws: [np.float32(13.52), np.float32(17.55), np.float32(25.92), np.float32(16.58), np.float32(22.14)]%

--- STEP 4: THE THRESHOLD TEST ---
✅ PASSED: Section 34 (Score: 38.970001220703125% | Needed: 37.0%)

--- AI PREDICTIONS ---
Law: Section 34 | Confidence: 38.97%


In [ ]:
!zip -r expert_ipc_brain.zip ./expert_ipc_brain

  adding: expert_ipc_brain/ (stored 0%)
  adding: expert_ipc_brain/checkpoint-617/ (stored 0%)
  adding: expert_ipc_brain/checkpoint-617/scheduler.pt (deflated 61%)
  adding: expert_ipc_brain/checkpoint-617/trainer_state.json (deflated 58%)
  adding: expert_ipc_brain/checkpoint-617/optimizer.pt (deflated 18%)
  adding: expert_ipc_brain/checkpoint-617/scaler.pt (deflated 64%)
  adding: expert_ipc_brain/checkpoint-617/rng_state.pth (deflated 26%)
  adding: expert_ipc_brain/checkpoint-617/model.safetensors (deflated 7%)
  adding: expert_ipc_brain/checkpoint-617/training_args.bin (deflated 53%)
  adding: expert_ipc_brain/ipc_classes.npy (deflated 50%)
  adding: expert_ipc_brain/tokenizer.json (deflated 71%)
  adding: expert_ipc_brain/checkpoint-1234/ (stored 0%)
  adding: expert_ipc_brain/checkpoint-1234/scheduler.pt (deflated 61%)
  adding: expert_ipc_brain/checkpoint-1234/trainer_state.json (deflated 63%)
  adding: expert_ipc_brain/checkpoint-1234/optimizer.pt (deflated 18%)
  adding: ex

In [ ]:
from google.colab import drive
import shutil

# 1. Connect Colab to your Google Drive
drive.mount('/content/drive')

# 2. Define where you want to save it in your Drive
# This creates a folder named "My_AI_Project" in your main Google Drive
destination_path = '/content/drive/MyDrive/My_AI_Project_Brain'

# 3. Copy the entire expert brain folder straight to Google Drive
shutil.copytree('./expert_ipc_brain', destination_path, dirs_exist_ok=True)

print(f"SUCCESS! The entire Brain has been securely copied to your Google Drive at: {destination_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SUCCESS! The entire Brain has been securely copied to your Google Drive at: /content/drive/MyDrive/My_AI_Project_Brain


In [ ]:
# Insert this right after: dataset = load_dataset(...)
real_ipc_names = dataset.features['labels'].feature.names
mapping_dict = {str(i): name for i, name in enumerate(real_ipc_names)}
df['labels'] = df['labels'].apply(lambda lst: [mapping_dict.get(str(law), str(law)) for law in lst])

In [ ]:
import numpy as np

# Load the classes file you saved
classes = np.load("./expert_ipc_brain/ipc_classes.npy", allow_pickle=True)

# Print the full list
print("--- THE AI'S ANSWER KEY ---")
for i, name in enumerate(classes):
    print(f"Index {i} = {name}")

--- THE AI'S ANSWER KEY ---
Index 0 = 0
Index 1 = 1
Index 2 = 10
Index 3 = 11
Index 4 = 12
Index 5 = 13
Index 6 = 14
Index 7 = 15
Index 8 = 16
Index 9 = 17
Index 10 = 18
Index 11 = 19
Index 12 = 2
Index 13 = 20
Index 14 = 21
Index 15 = 22
Index 16 = 23
Index 17 = 24
Index 18 = 25
Index 19 = 26
Index 20 = 27
Index 21 = 28
Index 22 = 29
Index 23 = 3
Index 24 = 30
Index 25 = 31
Index 26 = 32
Index 27 = 33
Index 28 = 34
Index 29 = 35
Index 30 = 36
Index 31 = 37
Index 32 = 38
Index 33 = 39
Index 34 = 4
Index 35 = 40
Index 36 = 41
Index 37 = 42
Index 38 = 43
Index 39 = 44
Index 40 = 45
Index 41 = 46
Index 42 = 47
Index 43 = 48
Index 44 = 49
Index 45 = 5
Index 46 = 50
Index 47 = 51
Index 48 = 52
Index 49 = 53
Index 50 = 54
Index 51 = 55
Index 52 = 56
Index 53 = 57
Index 54 = 58
Index 55 = 59
Index 56 = 6
Index 57 = 60
Index 58 = 61
Index 59 = 62
Index 60 = 63
Index 61 = 64
Index 62 = 65
Index 63 = 66
Index 64 = 67
Index 65 = 68
Index 66 = 69
Index 67 = 7
Index 68 = 70
Index 69 = 71
Index 70 =

In [ ]:
from datasets import load_dataset

# Load just the dataset's dictionary
dataset = load_dataset("Exploration-Lab/IL-TUR", "lsi", split='train[:1]')
real_ipc_names = dataset.features['labels'].feature.names

# Reveal what Label 13 actually is!
print(f"The AI was predicting: {real_ipc_names[13]}")

The AI was predicting: Section 148


In [ ]:
import numpy as np
from datasets import load_dataset

print("1. Fetching the real dictionary from Hugging Face...")
dataset = load_dataset("Exploration-Lab/IL-TUR", "lsi", split='train[:1]')
real_ipc_names = dataset.features['labels'].feature.names

print("2. Loading your alphabetical answer key...")
old_classes = np.load("./expert_ipc_brain/ipc_classes.npy", allow_pickle=True)

print("3. Translating the numbers into real IPC laws...")
# This converts '13' into its real name, '0' into its real name, etc.
# If it sees "IPC_OTHER", it just leaves it as "IPC_OTHER".
real_classes = [real_ipc_names[int(c)] if c.isdigit() else c for c in old_classes]

print("4. Overwriting the answer key...")
np.save("./expert_ipc_brain/ipc_classes.npy", real_classes)

print("\nSUCCESS! The Answer Key is fixed.")
print("Here are the first 10 laws your AI now knows:")
print(real_classes[:10])

1. Fetching the real dictionary from Hugging Face...
2. Loading your alphabetical answer key...
3. Translating the numbers into real IPC laws...
4. Overwriting the answer key...

SUCCESS! The Answer Key is fixed.
Here are the first 10 laws your AI now knows:
['Section 2', 'Section 3', 'Section 120B', 'Section 143', 'Section 147', 'Section 148', 'Section 149', 'Section 155', 'Section 156', 'Section 161']


In [ ]:
import numpy as np
from datasets import load_dataset

print("1. Fetching the real dictionary from Hugging Face...")
dataset = load_dataset("Exploration-Lab/IL-TUR", "lsi", split='train[:1]')
real_ipc_names = dataset.features['labels'].feature.names

print("2. Loading your alphabetical answer key...")
old_classes = np.load("./expert_ipc_brain/ipc_classes.npy", allow_pickle=True)

print("3. Formatting strings to use 'IPC' instead of 'Section'...")
formatted_classes = []
for c in old_classes:
    if c.isdigit():
        # Get the real name (e.g., "Section 302")
        raw_name = real_ipc_names[int(c)]

        # Replace "Section " with "IPC " (e.g., "IPC 302")
        formatted_name = raw_name.replace("Section ", "IPC ")
        formatted_classes.append(formatted_name)
    else:
        # If it's already a string like "IPC_OTHER", keep it
        formatted_classes.append(c)

print("4. Overwriting the answer key...")
np.save("./expert_ipc_brain/ipc_classes.npy", formatted_classes)

print("\nSUCCESS! The Answer Key is fixed.")
print("Here are the first 10 laws your AI now knows:")
print(formatted_classes[:10])

1. Fetching the real dictionary from Hugging Face...
2. Loading your alphabetical answer key...
3. Formatting strings to use 'IPC' instead of 'Section'...
4. Overwriting the answer key...

SUCCESS! The Answer Key is fixed.
Here are the first 10 laws your AI now knows:
[np.str_('Section 2'), np.str_('Section 3'), np.str_('Section 120B'), np.str_('Section 143'), np.str_('Section 147'), np.str_('Section 148'), np.str_('Section 149'), np.str_('Section 155'), np.str_('Section 156'), np.str_('Section 161')]
